# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata name and description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, their IDs, and the fields/columns within each. All references use the `@id` identifiers from the Croissant schema.

In [ ]:
# Retrieve all record sets defined in the dataset
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets detected in metadata.')
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}")
        if 'field' in rs and isinstance(rs['field'], list):
            print("  Fields:")
            for f in rs['field']:
                if isinstance(f, dict):
                    print(f"    {f.get('@id', str(f))}")
                else:
                    print(f"    {str(f)}")
        print()

## 3. Data Extraction
Load records from each record set into DataFrames for analysis. Use the record set and field `@id`s from the overview above.
If there is only one record set, load it directly; else, extract data for all.

In [ ]:
# Gather record set @ids
record_set_ids = [rs["@id"] for rs in dataset.record_sets]
print('Record set ids:', record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    # For each record set, load records as a DataFrame
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f'Loaded {len(records)} records for record set {rs_id}')
    if len(records) > 0:
        print('Columns:', dataframes[rs_id].columns.tolist())
        display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic analyses: filtering, normalization, and grouping.
To do this, select a numeric field (`@id` as shown above) for numeric operations and another field for grouping where appropriate.

In [ ]:
# Pick a record set to analyze (using the first record set if there's only one)
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
else:
    print('No record sets available for EDA!')

# Display all available columns for reference
print('Columns in DataFrame:', list(df.columns))

# Suggest a numeric field (@id) for demo (update if schema changes):
# For the FAIR^2 dataset variables might include: 'age' (likely @id: 'schema:age' or similar)
numeric_field_candidates = [c for c in df.columns if ('age' in c.lower() or 'interval' in c.lower() or 'count' in c.lower())]
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f'Numeric field selected: {numeric_field}')
else:
    # Fallback: try the first float/int column
    num_types = ["float64", "int64"]
    for col in df.columns:
        if df[col].dtype.name in num_types:
            numeric_field = col
            print(f'Numeric field selected by type: {numeric_field}')
            break
    else:
        raise Exception("No suitable numeric field found!")

# Demonstrate threshold filtering
try:
    threshold = float(df[numeric_field].mean()) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field for filtered records
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
except Exception as e:
    print('Numeric field processing error:', str(e))

# Choose a group field (@id) for grouping (e.g., sex, anatomical site, status)
group_field_candidates = [c for c in df.columns if (('sex' in c.lower()) or ('site' in c.lower()) or ('status' in c.lower()) or ('type' in c.lower()))]
if group_field_candidates:
    group_field = group_field_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
else:
    print('No suitable group field found.')

## 5. Visualization
Visualize the distribution of the numeric field (e.g., age, diagnosis interval) and the grouped summary by group field.
Uses `matplotlib` for basic plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouped_df exists, make a bar plot
if 'grouped_df' in locals():
    plt.figure(figsize=(8,5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to access, examine, and summarize the FAIR² dataset using `mlcroissant`.

- We explored available record sets and fields via their `@id` identifiers.
- Data was loaded into DataFrames with references preserved.
- We performed basic numeric filtering, normalization, grouping, and summary visualization.

**Next steps:** You can adapt the notebook to perform more domain-specific analyses or data preparation for your ML or statistical workflow.